In [ ]:
import os
import json
from typing import Literal

from dotenv import load_dotenv
from openai import OpenAI

from helpers.prompt_maker import get_links_system_prompt, get_links_user_prompt
from helpers.scraper import fetch_website_contents

In [26]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

anthropic_url = "https://api.anthropic.com/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI(api_key=openai_api_key)
anthropic = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [27]:
def define_model_and_provider(
    model_type: Literal["gpt", "claude", "llama"],
) -> tuple[str, OpenAI]:
    if model_type.lower() == "gpt":
        model = "gpt-5.6-luna"
        provider = openai
    elif model_type.lower() == "claude":
        model = "claude-sonnet-5"
        provider = anthropic
    else:
        model = "llama3.2"
        provider = ollama
    return model, provider

In [ ]:
from pydantic import BaseModel, ConfigDict


class Link(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: str
    url: str


class RelevantLinks(BaseModel):
    model_config = ConfigDict(extra="forbid")
    links: list[Link]


LINKS_JSON_SCHEMA = {
    "name": "relevant_links",
    "strict": True,
    "schema": RelevantLinks.model_json_schema(),
}


def message_llm(
    system_prompt: str,
    user_prompt: str,
    model_type: Literal["gpt", "claude", "llama"] = "gpt",
    json=False,
) -> str:
    model, provider = define_model_and_provider(model_type)
    if json is True:
        json_response_format = (
            {"type": "json_schema", "json_schema": LINKS_JSON_SCHEMA}
            if model_type == "claude"
            else {"type": "json_object"}
        )
    else:
        json_response_format = {"type": "text"}
    response = provider.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=json_response_format
    )
    result = response.choices[0].message.content or ""
    return result

In [ ]:
def select_relevant_links(url: str, model_type: Literal["gpt", "claude", "llama"]):
    system_prompt = get_links_system_prompt()
    user_prompt = get_links_user_prompt(url)
    result = message_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        model_type=model_type,
        json=True,
    )
    links = json.loads(result)
    return links

In [32]:
def fetch_page_and_all_relevant_links(
    url: str, model_type: Literal["gpt", "claude", "llama"]
):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url=url, model_type=model_type)
    result = f"## Landing Page:\n\n{contents}\n\nRelevant Links:"
    for link in relevant_links["links"]:
        result += f"\n\n###Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [35]:
print(fetch_page_and_all_relevant_links(url="https://www.anthropic.com/", model_type="llama"))

## Landing Page:

Home \ Anthropic

Skip to main content
Skip to footer
Research
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
About Claude
Overview
Pricing
Contact sales
Models
Mythos
Fable
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claude
Log in to Claude
Download app
Download app
Download app
Research
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Tr